# Sleep-EDF 睡眠分期：SNN 与动力学分析

本 Notebook 是端到端实验骨架。首次运行会通过 MNE 下载公开的 Sleep-EDF 数据；当前默认仅使用少量受试者，结果用于流程演示，不代表完整数据集结论。

## 1. 环境与可复现设置

In [ ]:
from pathlib import Path
import sys
import random
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import ConfusionMatrixDisplay

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from snn_eeg.data import load_sleep_edf
from snn_eeg.encoding import rate_encode
from snn_eeg.models import TwoLayerSNN
from snn_eeg.baselines import make_baselines
from snn_eeg.metrics import classification_metrics, spike_activity

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE

## 2. 获取并预处理数据：MNE 滤波、epoch 与频带功率

In [ ]:
# 可将 subjects 扩大；首次运行需要网络和本地存储空间。
X, y_text, groups = load_sleep_edf(subjects=(0, 1), recording=1)
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_text)
print('特征矩阵:', X.shape, '| 类别:', dict(enumerate(label_encoder.classes_)))

## 3. 按受试者划分，避免数据泄漏

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
train_idx, test_idx = next(splitter.split(X, y, groups))
scaler = StandardScaler().fit(X[train_idx])
X_train = scaler.transform(X[train_idx]).astype(np.float32)
X_test = scaler.transform(X[test_idx]).astype(np.float32)
y_train, y_test = y[train_idx], y[test_idx]
print('训练受试者:', np.unique(groups[train_idx]), '| 测试受试者:', np.unique(groups[test_idx]))

## 4. SVM 与普通 MLP 基线

In [ ]:
baseline_results = {}
for name, model in make_baselines(SEED).items():
    model.fit(X_train, y_train)
    prediction = model.predict(X_test)
    baseline_results[name] = classification_metrics(y_test, prediction)
    print(name, 'accuracy=', round(baseline_results[name]['accuracy'], 4),
          'macro-F1=', round(baseline_results[name]['macro_f1'], 4))

## 5. 特征到脉冲：Bernoulli rate coding

In [ ]:
N_STEPS = 32
X_train_tensor = torch.tensor(X_train, device=DEVICE)
X_test_tensor = torch.tensor(X_test, device=DEVICE)
train_generator = torch.Generator(device=DEVICE).manual_seed(SEED)
test_generator = torch.Generator(device=DEVICE).manual_seed(SEED + 1)
train_spikes = rate_encode(X_train_tensor, N_STEPS, train_generator)
test_spikes = rate_encode(X_test_tensor, N_STEPS, test_generator)
train_spikes.shape

## 6. 两层 LIF SNN 训练

In [ ]:
y_train_tensor = torch.tensor(y_train, dtype=torch.long, device=DEVICE)
y_test_tensor = torch.tensor(y_test, dtype=torch.long, device=DEVICE)
snn_model = TwoLayerSNN(X_train.shape[1], n_hidden=64,
                        n_classes=len(label_encoder.classes_)).to(DEVICE)
optimizer = torch.optim.Adam(snn_model.parameters(), lr=1e-3)
loss_fn = torch.nn.CrossEntropyLoss()
EPOCHS = 20

for epoch in range(EPOCHS):
    snn_model.train()
    optimizer.zero_grad()
    output_counts, _ = snn_model(train_spikes)
    loss = loss_fn(output_counts, y_train_tensor)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch + 1:02d} | loss={loss.item():.4f}')

## 7. 分类指标与混淆矩阵

In [ ]:
snn_model.eval()
with torch.no_grad():
    snn_counts, activity = snn_model(test_spikes)
    snn_prediction = snn_counts.argmax(dim=1).cpu().numpy()
snn_results = classification_metrics(y_test, snn_prediction)
print('SNN accuracy:', round(snn_results['accuracy'], 4))
print('SNN macro-F1:', round(snn_results['macro_f1'], 4))
print(snn_results['report'])
ConfusionMatrixDisplay(snn_results['confusion_matrix'],
                       display_labels=label_encoder.classes_).plot(xticks_rotation=45)
plt.tight_layout()

## 8. 脉冲动力学与活动量代理

In [ ]:
hidden_spikes = activity['hidden_spikes'].cpu().numpy()
output_spikes = activity['output_spikes'].cpu().numpy()
print('隐藏层活动:', spike_activity(hidden_spikes))
print('输出层活动:', spike_activity(output_spikes))

example_index = 0
plt.figure(figsize=(10, 4))
plt.imshow(hidden_spikes[:, example_index, :].T, aspect='auto',
           origin='lower', interpolation='nearest', cmap='Greys')
plt.xlabel('时间步')
plt.ylabel('隐藏神经元')
plt.title('测试样本隐藏层脉冲栅格')
plt.tight_layout()

# 脉冲数是活动量代理，不是焦耳或实测功耗。

## 9. 结果记录与解释

汇总 SVM、MLP、SNN 的 Accuracy 和 Macro-F1，并结合混淆矩阵讨论类别不平衡、受试者泛化和随机脉冲编码的影响。记录数据子集、受试者划分、随机种子、时间步数、训练轮数和软件/硬件环境。脉冲数只能作为活动量代理；没有硬件测量或经过校准的能耗模型时，不得据此声称获得了真实能耗节省。

后续 OpenNeuro 听觉/嗅觉扩展需要先选定并核实数据集 ID、任务事件、标签和许可，再实现新的数据适配器。